# 04 · Protein Folding: Gradients in 3D Space
### *Using NeRF and NMR data to fold a protein backbone*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins-lab/diff-biophys/blob/main/examples/interactive_tutorials/04_protein_folding_nerf.ipynb)

---

**What you will learn:**
1. How to build 3D atomic structures from torsion angles using the **NeRF** algorithm.
2. How to use **Residual Dipolar Couplings (RDCs)** and **Chemical Shifts** as structural constraints.
3. How to "fold" a random coil into an alpha-helix using nothing but gradients.

**Time:** ~20 minutes

## 0 · Setup

Install the library and optimization tools.

In [ ]:
# Install diff-biophys and optax
%pip install -q diff-biophys==0.1.6 optax matplotlib

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
from diff_biophys.geometry.nerf import chain_nerf
from diff_biophys.nmr.rdc import calculate_rdc, calculate_q_factor
from diff_biophys.nmr.chemical_shifts import predict_ca_shifts

jax.config.update("jax_enable_x64", False) # float32 is faster and sufficient

## 1 · Create a Target "Alpha Helix"

We'll define a target 10-residue alpha-helix ($\phi \approx -57^\circ, \psi \approx -47^\circ$) and compute its NMR observables.

In [ ]:
N_RES = 10

# Ideal backbone parameters (Engh & Huber 1991)
CA_C_LENGTH = 1.525
C_N_LENGTH = 1.329
N_CA_LENGTH = 1.459
CA_C_N_ANGLE = jnp.radians(116.2)
C_N_CA_ANGLE = jnp.radians(121.7)
N_CA_C_ANGLE = jnp.radians(111.2)

bond_lengths = jnp.array([C_N_LENGTH, N_CA_LENGTH, CA_C_LENGTH] * (N_RES - 1))
bond_angles = jnp.array([CA_C_N_ANGLE, C_N_CA_ANGLE, N_CA_C_ANGLE] * (N_RES - 1))

# Seed atoms (N0, CA0, C0)
init_coords = jnp.array([
    [0.0, 0.0, 0.0],
    [N_CA_LENGTH, 0.0, 0.0],
    [N_CA_LENGTH + CA_C_LENGTH * jnp.cos(jnp.pi - N_CA_C_ANGLE),
     CA_C_LENGTH * jnp.sin(jnp.pi - N_CA_C_ANGLE), 0.0]
])

def build_peptide(phi, psi):
    # Trans peptide bonds (omega = 180)
    omega = jnp.full((N_RES - 1,), jnp.pi)
    
    # Interleave dihedrals for NeRF: psi[0], omega[0], phi[1], psi[1], omega[1], phi[2]...
    # This matches the backbone connectivity N-CA-C-N-CA-C...
    d = jnp.stack([psi[:-1], omega, phi[1:]], axis=1).ravel()
    
    return chain_nerf(init_coords, bond_lengths, bond_angles, d)

# Target: Ideal Alpha Helix
target_phi = jnp.full((N_RES,), jnp.deg2rad(-57.0))
target_psi = jnp.full((N_RES,), jnp.deg2rad(-47.0))
target_coords = build_peptide(target_phi, target_psi)

print(f"Target structure created with {len(target_coords)} atoms.")

## 2 · Synthetic NMR Data

We'll use the **Residual Dipolar Couplings (RDCs)** from the peptide bonds as our primary structural constraint. RDCs tell us about the *orientation* of each bond vector relative to an external alignment frame.

In [ ]:
def get_bond_vectors(coords):
    # Extract C-N vectors (indices 2->3, 5->6, etc.)
    c_atoms = coords[2::3][:-1]
    n_atoms = coords[3::3]
    vecs = n_atoms - c_atoms
    return vecs / jnp.linalg.norm(vecs, axis=-1, keepdims=True)

target_vectors = get_bond_vectors(target_coords)

# Generate synthetic RDCs (Axial Da=10Hz, Rhombicity R=0.2)
target_rdcs = calculate_rdc(target_vectors, da=10.0, r=0.2)

# Generate synthetic Chemical Shifts
rc_shifts = jnp.full((N_RES,), 52.5) # Alanine random coil
target_shifts = predict_ca_shifts(target_phi, target_psi, rc_shifts)

print(f"Target RDCs (Hz): {target_rdcs[:3]}...")
print(f"Target Cα Shifts (ppm): {target_shifts[:3]}...")

## 3 · "Fold" from a Random Coil

Now the magic: we start from a random set of $\phi, \psi$ angles and use gradient descent to recover the alpha-helix.

In [ ]:
def loss_fn(params):
    phi, psi = params
    coords = build_peptide(phi, psi)
    
    # 1. RDC Loss (Q-factor)
    curr_vectors = get_bond_vectors(coords)
    curr_rdcs = calculate_rdc(curr_vectors, da=10.0, r=0.2)
    q_loss = calculate_q_factor(curr_rdcs, target_rdcs)
    
    # 2. Chemical Shift Loss
    curr_shifts = predict_ca_shifts(phi, psi, rc_shifts)
    shift_loss = jnp.mean((curr_shifts - target_shifts)**2)
    
    return q_loss + 0.5 * shift_loss

# Initialise with random angles
key = jax.random.PRNGKey(42)
phi_init = jax.random.uniform(key, (N_RES,), minval=-jnp.pi, maxval=jnp.pi)
psi_init = jax.random.uniform(key, (N_RES,), minval=-jnp.pi, maxval=jnp.pi)
params = (phi_init, psi_init)

optimizer = optax.adam(learning_rate=0.05)
opt_state = optimizer.init(params)

@jax.jit
def step(params, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

print("Starting optimization...")
losses = []
for i in range(101):
    params, opt_state, loss_value = step(params, opt_state)
    losses.append(loss_value)
    if i % 20 == 0:
        print(f"Step {i:03d} | Loss: {loss_value:.4f}")

## 4 · Visualise the Result

Let's look at the loss curve and the final 3D coordinates.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.title("Folding Loss (RDC + CS)")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

# Get final coordinates
final_phi, final_psi = params
final_coords = build_peptide(final_phi, final_psi)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot Target (dashed)
ax.plot(target_coords[:, 0], target_coords[:, 1], target_coords[:, 2], 'k--', label="Target (Alpha Helix)", alpha=0.5)
# Plot Final (solid)
ax.plot(final_coords[:, 0], final_coords[:, 1], final_coords[:, 2], 'r-', label="Optimised Backbone", linewidth=3)

ax.set_title("Differentiable Folding Result")
ax.legend()
plt.show()